In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [1]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [14]:
import langchain_google_genai
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")
class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False, #approval not required for this tool
                "send_email": True, #approval required
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [15]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John." #state chaining
    },
    config=config
)

In [16]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

In [17]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='f017fed745bf1bed9760b3ec5148915f')]


In [18]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.


## Approve

In [19]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='d1e6c46b-def0-4dbe-8083-e424d88762ef'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'c65260ae-2f39-4296-9a4f-11bfafb7607a': 'EjQKMgG+Pvb7QCPsDz3no44uL/YiDkmlxC1gdgVFfQ9V/61x5Cu22+uxqIYHWw9BIDy5GBvb'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d020a-94d9-7871-9c90-988400183e1f-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'c65260ae-2f39-4296-9a4f-11bfafb7607a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tok

## Reject

In [26]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='d1e6c46b-def0-4dbe-8083-e424d88762ef'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'c65260ae-2f39-4296-9a4f-11bfafb7607a': 'EjQKMgG+Pvb7QCPsDz3no44uL/YiDkmlxC1gdgVFfQ9V/61x5Cu22+uxqIYHWw9BIDy5GBvb'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d020a-94d9-7871-9c90-988400183e1f-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'c65260ae-2f39-4296-9a4f-11bfafb7607a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tok

In [27]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [28]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='d1e6c46b-def0-4dbe-8083-e424d88762ef'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'c65260ae-2f39-4296-9a4f-11bfafb7607a': 'EjQKMgG+Pvb7QCPsDz3no44uL/YiDkmlxC1gdgVFfQ9V/61x5Cu22+uxqIYHWw9BIDy5GBvb'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d020a-94d9-7871-9c90-988400183e1f-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'c65260ae-2f39-4296-9a4f-11bfafb7607a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tok